# Policy Gradient Hello World

## 1. Goal

Build one minimal, shared experiment to understand the evolution of:

**REINFORCE → RLOO → PPO → GRPO → Dr. GRPO → DAPO → CISPO → GSPO**

For each algorithm, answer three questions:

1. **Intuition:** What problem does it solve, and why is its core idea natural?
2. **Old vs. New:** Which parts of the previous algorithm remain unchanged? What exactly changed?
3. **Theory vs. Empirical:** What can be mathematically derived or proved, and what is only supported by experiments?

**Core principle:** Keep the dataset fixed. Change one algorithmic component at a time.

---

## 2. Assumptions

To keep the experiment minimal:

- Each character represents one token, including digits.
- Each prompt produces exactly 3 sampled completions.
- Each completion has one final sequence-level reward.
- All completions are sampled independently from the same frozen rollout policy, conditioned on their prompt.
- `...` represents omitted tokens, not literal ellipsis tokens.
- Token lengths include the final answer tokens but exclude prompt tokens and EOS.
- Rewards initially depend only on correctness or the specified grading rubric.
- Initially, disable KL regularization: `beta = 0`.
- No learned value function is needed initially.

We distinguish three concepts:

| Symbol | Meaning |
|---|---|
| `T` | Completion length in tokens |
| `R` | Reward assigned to the entire completion |
| `A` | Advantage used to update the policy |

The **reward evaluator** reads the complete response and assigns `R`. The policy-gradient algorithm converts `R` into `A` and uses it to update token probabilities.

---

## 3. Dataset

### Q1 — Short Correct, Long Incorrect

**Prompt:** `2 + 2 = ?`

| Response | Generated completion | Tokens | Reward |
|---|---|---:|---:|
| A1 | `abdcef...4` | 10 | 1 |
| A2 | `fijklm...6` | 30 | 0 |
| A3 | `tyuiioo...10` | 100 | 0 |

```text
Lengths = [10, 30, 100]
Rewards = [1, 0, 0]
```

**Purpose:** Study how algorithms reinforce a short correct answer and suppress longer incorrect ones.

### Q2 — Short Incorrect, Long Correct

**Prompt:** `7 × 8 = ?`

| Response | Generated completion | Tokens | Reward |
|---|---|---:|---:|
| B1 | `abc...54` | 50 | 0 |
| B2 | `def...49` | 200 | 0 |
| B3 | `ghi...56` | 500 | 1 |

```text
Lengths = [50, 200, 500]
Rewards = [0, 0, 1]
```

**Purpose:** Reverse Q1. Study whether the algorithm can reinforce a long correct answer without inadvertently favoring short completions.

### Q3 — All Correct

**Prompt:** `3 + 5 = ?`

| Response | Generated completion | Tokens | Reward |
|---|---|---:|---:|
| C1 | `abc...8` | 50 | 1 |
| C2 | `def...8` | 200 | 1 |
| C3 | `ghi...8` | 500 | 1 |

```text
Lengths = [50, 200, 500]
Rewards = [1, 1, 1]
```

**Purpose:** Test zero-variance groups and determine whether the algorithm can learn to prefer shorter answers when all answers are equally correct.

### Q4 — All Incorrect

**Prompt:** `9 - 5 = ?`

| Response | Generated completion | Tokens | Reward |
|---|---|---:|---:|
| D1 | `abc...3` | 50 | 0 |
| D2 | `def...6` | 200 | 0 |
| D3 | `ghi...1` | 500 | 0 |

```text
Lengths = [50, 200, 500]
Rewards = [0, 0, 0]
```

**Purpose:** Test the absence of positive examples, zero-variance advantages, and dynamic sampling.

### Q5 — Partial Credit

**Prompt:** `8 / 2 = ?` (A grading rubric also evaluates reasoning.)

| Response | Evaluation | Tokens | Reward |
|---|---|---:|---:|
| E1 | Correct answer and reasoning | 10 | 1.0 |
| E2 | Correct final answer, flawed reasoning | 30 | 0.5 |
| E3 | Incorrect answer | 100 | 0.0 |

```text
Lengths = [10, 30, 100]
Rewards = [1.0, 0.5, 0.0]
```

**Purpose:** Investigate how group standard deviation changes advantage scaling and why Dr. GRPO removes that normalization.

This is a deliberately synthetic reward rubric, not a claim that all verifiers award partial credit.

---